<a href="https://colab.research.google.com/github/guilhermelaviola/NaturalLanguageProcessing/blob/main/Class14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NLP Applications in Chatbots & Virtual Assistants**
Natural Language Processing (NLP) is a rapidly evolving field driven by powerful language models like GPT, enabling applications such as chatbots, translation, and summarization that interact with users in natural and personalized ways. While these models offer significant potential, they are purely statistical tools and raise important ethical concerns, including bias, lack of transparency, environmental impact, and reliability—especially in sensitive domains like health and justice. Approaches such as Prompt Engineering and Retrieval Augmented Generation (RAG) help improve accuracy and trustworthiness by guiding model behavior and connecting it to reliable external information. As NLP continues to advance and play an increasingly central role in daily life, its development must be guided by ethical principles, interdisciplinary collaboration, and responsible practices to ensure fairness, sustainability, and broad societal benefit.

In [12]:
! pip3 install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=eebed5f7e6e8b1b948fe8680b9ea5c9d01a3611448e7b83b1df43d7760955fc0
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [13]:
# Importing all the necessary libraries resources:
import random
from getpass import getpass
from transformers import pipeline
import wikipedia

## **Example: Rule-based Chatbot**
This simple example demonstrates how a chatbot can respond to specific commands with predefined, but it's important to highlight that more complex systems use deep learning models to understand and generate more natural and contextual responses.

In [14]:
# Defining the database instances:
customers = {
    '1': {
        'balance': 500.00,
        'name': 'John Smith',
        'password': 'johnpass'
    },
    '2': {
        'balance': 900.00,
        'name': 'Jane Anderson',
        'pass': 'janepass'
    }
}

In [15]:
# Defining the start function:
def start():
  welcome = [
      'Hello! Welcome to MyBank. How can I help you?'
  ]

  options = [
      ['1 - Log In', login],
      ['2 - Create an account', create_account],
      ['3 - End chat', end_chat]
  ]

  print('\n' + random.choice(welcome))

  for option in options:
    print(option[0])
    choice = input('Pick a number: ')
    options[int(choice)-1][1]()



In [16]:
# Defining the function to create an account:
def create_account():
    print('\nWe are not accepting new account applications at this time. We will let you know when we are open for new applications again!')

In [17]:
# Defining the function to Log In:
def login():
  customer = input('\nWhat is your ID? ').replace('.', '').replace('-', '')

  # Check should have been done via API:
  if not customer in customers:
    print('\nCustomer not found.')
    start()
    password = getpass('\nWhat is your password? ')

  # Check shall be done via API:
  if password == customers[customer]['password']:
    print('\nCustomers authenticated successfully.')
    logged(customers[customer])
  else:
    print('\nWrong password, Try again.')
    start()

In [18]:
# Defining the function for the application to interact with the user once logged in:
def logged(customer):
  print(f'\nHello, {customer['name']}. Your current balance is US${customer['balance']}. What would you like to do?')

  options = [
        ['1 - Transfer money', transfer],
        ['2 - End chat', end_chat]
    ]

  for option in options:
    print(option[0])
  choice = input('Pick a number: ')
  options[int(choice)-1][1](customer)

In [19]:
# Defining the function to transfer money:
def transfer(customer):
  value = float(input('\nHow much would you like to transfer? US$ '))
  receiver = input(f'\nenter the ID of the person who is going to transfer US${value}: ')

  # Check shall be done via API:
  receiver_name = customers[receiver]['name']

  confirmation = input(f'\nAre you sure you want to transfer US${value} ti {receiver_name}? (YES or NO): ')
  if confirmation.lower().strip() == 'yes':
    # Operation shall be done via API:
    customer['balance'] -= value
    customer[receiver]['balance'] += value

    print('\nTransfer done successfully!')
    logged(customer)
  else:
    logged(customer)

In [20]:
# Defining the function to end the chat:
def end_chat(customere=None):
  end = [
      'See you!',
      'Hope to see you soon!',
      'I hope you liked our service, bye!'
    ]

  print('\n' + random.choice(end))
  return

In [21]:
# Thesting the chatbot:
start()


Hello! Welcome to MyBank. How can I help you?
1 - Log In
Pick a number: 1

What is your ID? 1234

Customer not found.

Hello! Welcome to MyBank. How can I help you?
1 - Log In
Pick a number: 1


KeyboardInterrupt: Interrupted by user

## **Example: Chatbot for Sentiment Analysis**

In [22]:
# Developing a chatbot for sentiment analysis:
classifier = pipeline(
    model='lxyuan/distilbert-base-multilingual-cased-sentiments-student',
)

wikipedia.set_lang('en')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cpu


In [24]:
def start():
  question = input('Ask anything:\n')
  results = wikipedia.search(question)
  best_result = results[0]
  text = wikipedia.page(best_result)
  print('\n' + text.summary)
  print(f'\nSourcee: {best_result} - Wikipedia')
  feedback = input('\nWhst do you think of my response?\n')
  sentiment = classifier(feedback)

  responses = [
      'I am happy you liked my response!',
      'Thank you for your feedback. Hope to hear from you again soon.',
      'I am sorry you did not like my response. Would you like to try another question?'
  ]

  if sentiment[0]['label'] == 'positive' and sentiment[0]['score'] > 0.6:
    print('\n' + responses[0])
  elif sentiment[0]['label'] == 'negative' and sentiment[0]['score'] > 0.6:
    print('\n' + responses[2])
  else:
    print('\n' + responses[1])
  return sentiment

In [25]:
# Testing the chatbot:
start()

Ask anything:
Who composed the song Fascination from the album 'Young Americans' by David Bowie?

"Changes" is a song by the English musician David Bowie from his 1971 album Hunky Dory. RCA Records then released it as a single from the album on 7 January 1972. Written following his promotional tour of America in early 1971, "Changes" was recorded at Trident Studios in London in July that year. Co-produced by Bowie and Ken Scott, it featured Rick Wakeman on piano and the musicians who would later become known as the Spiders from Mars—Mick Ronson, Trevor Bolder and Mick Woodmansey.
At this point in his career, Bowie had experimented with numerous musical styles, all of which failed to earn him stardom. The lyrics of "Changes" reflect this, with the first verse focusing on the compulsive nature of artistic reinvention and distancing oneself from the rock mainstream. The second verse concerns clashes between children and their parents, urging them to allow their children to be themselves a

[{'label': 'negative', 'score': 0.8431380987167358}]